In [7]:
from transformers import pipeline
sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")
print(sentiment_analysis("hello i hate you but love michael"))

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5511.35it/s]
RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'POSITIVE', 'score': 0.9962339997291565}]


In [6]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-base-zeroshot-v1")
sequence_to_classify = "google chrome is generally considered the best browser and you should use it."
candidate_labels = ["the text favours google-chrome", "the text strongly favours google-chrome"]
output = classifier(sequence_to_classify, candidate_labels, multi_label=False)
print(output)


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1540.12it/s]


{'sequence': 'google chrome is generally considered the best browser and you should use it.', 'labels': ['the text favours google-chrome', 'the text strongly favours google-chrome'], 'scores': [0.611382007598877, 0.38861802220344543]}


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

classifier_model_path = "teknology/ad-classifier-v0.3"
tokenizer = AutoTokenizer.from_pretrained(classifier_model_path)
model = AutoModelForSequenceClassification.from_pretrained(classifier_model_path)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

def classify_with_probs(passages):
    inputs = tokenizer(
        passages,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)

    rows = []
    for passage, row_probs in zip(passages, probs, strict=True):
        score_by_label = {
            model.config.id2label[i]: float(row_probs[i].item())
            for i in range(row_probs.shape[-1])
        }
        rows.append(
            {
                "text": passage,
                "pred_id": int(torch.argmax(row_probs).item()),
                "score_by_label": score_by_label,
            }
        )
    return rows

tests = [
    "The capital of France is Paris.",
    "Paris is the capital of France. If you're planning a trip, Booking.com has great deals on hotels.",
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "The iPhone 16 offers improved battery life and camera performance compared with older models.",
]

rows = classify_with_probs(tests)

for row in rows:
    print(row)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1586.55it/s]


id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
label2id: {'LABEL_0': 0, 'LABEL_1': 1}
{'text': 'The capital of France is Paris.', 'pred_id': 0, 'score_by_label': {'LABEL_0': 0.9959989786148071, 'LABEL_1': 0.004001013468950987}}
{'text': "Paris is the capital of France. If you're planning a trip, Booking.com has great deals on hotels.", 'pred_id': 1, 'score_by_label': {'LABEL_0': 0.08442148566246033, 'LABEL_1': 0.9155784845352173}}
{'text': 'Water boils at 100 degrees Celsius at standard atmospheric pressure.', 'pred_id': 0, 'score_by_label': {'LABEL_0': 0.9959431290626526, 'LABEL_1': 0.004056882578879595}}
{'text': 'The iPhone 16 offers improved battery life and camera performance compared with older models.', 'pred_id': 0, 'score_by_label': {'LABEL_0': 0.9972668886184692, 'LABEL_1': 0.0027331809978932142}}
